In [1]:
import torch
from torch import nn
from transformers import BertModel, BertTokenizer

In [4]:
model_name = 'bert-base-uncased'

In [5]:
model = BertModel.from_pretrained(model_name, output_hidden_states=True)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
tokenizer = BertTokenizer.from_pretrained(model_name)

### 1. input

In [8]:
sentence = "This GitLab reference architecture can help you deploy GitLab to up to 3,000 users, and then maintain uptime and access for those users. You can also use this architecture to provide improved GitLab uptime and availability for fewer than 3,000 users. For fewer users, reduce the stated node sizes as needed."

In [10]:
token_input = tokenizer(sentence, return_tensors="pt")

In [11]:
input_ids = token_input["input_ids"]

In [13]:
input_ids, input_ids.shape

(tensor([[  101,  2023, 21025, 19646,  7875,  4431,  4294,  2064,  2393,  2017,
          21296, 21025, 19646,  7875,  2000,  2039,  2000,  1017,  1010,  2199,
           5198,  1010,  1998,  2059,  5441,  2039,  7292,  1998,  3229,  2005,
           2216,  5198,  1012,  2017,  2064,  2036,  2224,  2023,  4294,  2000,
           3073,  5301, 21025, 19646,  7875,  2039,  7292,  1998, 11343,  2005,
           8491,  2084,  1017,  1010,  2199,  5198,  1012,  2005,  8491,  5198,
           1010,  5547,  1996,  3090, 13045, 10826,  2004,  2734,  1012,   102]]),
 torch.Size([1, 70]))

### 2. model forward
- forward
  - embedding => encoder => pooler

In [18]:
model.eval()
with torch.no_grad():
    outputs = model(**token_input)

### 3. output

- len(outputs) == 3

- outputs[0]

  - last_hidden_state, shape: batch_size*seq_len*hidden_size(1*22*768)
- outputs[1]

  - pooler_output, shape: batch_size*hidden_size(1*768)
  - Last layer hidden-state of the first token of the sequence (classification token, [CLS])
- outputs[2] (model.config.output_hidden_states = True)

  - type: tuple
  - one for the output of the embeddings(1), if the model has an embedding layer(12), + one for the output of each layer)
    - (1+12)*(batch_size*seq_len*hidden_size) = 13*1*22*768
  
- outputs[0] == outputs[2][-1]

- outputs[1] == model.pooler(outputs[2][-1])

- outputs[2][0] == model.embeddings(token_input['input_ids'], token_input['token_type_ids'])

In [20]:
len(outputs), type(outputs)

(3, transformers.modeling_outputs.BaseModelOutputWithPoolingAndCrossAttentions)

In [31]:
type(outputs[2]), len(outputs[2])  

# hidden_states 输出 长度13  1个embedding 层 + 12 attention layer 层 

(tuple, 13)

In [24]:
type(outputs[1]), len(outputs[1][0]) 

(torch.Tensor, 768)

In [25]:
outputs[2][0] 

tensor([[[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
         [-0.6485,  0.6739, -0.0932,  ...,  0.4475,  0.6696,  0.1820],
         [-0.3415, -0.4800, -0.9290,  ...,  0.0341, -0.4749,  0.0779],
         ...,
         [-0.1870, -0.1056,  0.4604,  ..., -0.1795,  1.0498,  1.1996],
         [-0.3697,  0.2690,  0.0202,  ...,  0.1602,  0.5930,  0.8433],
         [-0.3487,  0.1841,  0.2145,  ..., -0.4132, -0.0205,  0.2889]]])

In [30]:
 # 与 outputs[2][0]  输出相同
model.embeddings(token_input['input_ids'], token_input['token_type_ids'])

tensor([[[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
         [-0.6485,  0.6739, -0.0932,  ...,  0.4475,  0.6696,  0.1820],
         [-0.3415, -0.4800, -0.9290,  ...,  0.0341, -0.4749,  0.0779],
         ...,
         [-0.1870, -0.1056,  0.4604,  ..., -0.1795,  1.0498,  1.1996],
         [-0.3697,  0.2690,  0.0202,  ...,  0.1602,  0.5930,  0.8433],
         [-0.3487,  0.1841,  0.2145,  ..., -0.4132, -0.0205,  0.2889]]],
       grad_fn=<NativeLayerNormBackward0>)